# 01_ingest_raw.py 결과 확인

`data/raw` 아래 companies / prices / financials / dividends 원본 수집 결과를 확인한다.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_raw").getOrCreate()

# .cache()로 메모리에 적재 - 이후 셀들이 매번 parquet을 재스캔하지 않고 캐시를 재사용해 빨라짐
companies = spark.read.parquet("/opt/spark-apps/data/raw/companies").cache()
prices = spark.read.parquet("/opt/spark-apps/data/raw/prices").cache()
financials = spark.read.parquet("/opt/spark-apps/data/raw/financials").cache()
dividends = spark.read.parquet("/opt/spark-apps/data/raw/dividends").cache()

print(f"companies : {companies.count()}건")
print(f"prices    : {prices.count()}건")
print(f"financials: {financials.count()}건")
print(f"dividends : {dividends.count()}건")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/04 08:21:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


companies : 2555건


prices    : 1797973건


financials: 1539859건
dividends : 135587건


## 1. companies (KRX 상장종목 + DART corpCode 매핑)

In [2]:
companies.orderBy("stock_code").toPandas()

,corp_cls,corp_code,corp_name,induty_code,stock_code,bas_dt
0,Y,00119195,동화약품,212,000020,20260724
1,Y,00112378,KR모터스,319,000040,20260724
2,Y,00101628,경방,47119,000050,20260724
3,Y,00126937,삼양홀딩스,64992,000070,20260724
4,Y,00150244,하이트진로,11122,000080,20260724
...,...,...,...,...,...,...
2550,K,01416235,고스트스튜디오,58212,950190,20260724
2551,K,01442115,소마젠,73909,950200,20260724
2552,Y,01510489,프레스티지바이오파마,70113,950210,20260724
2553,K,01511558,네오이뮨텍,70130,950220,20260724


## 2. prices (날짜별 건수, snapshot_type별 분포)
`snapshot_type`: current(백테스트 대상 구간) / 1m_ago / 12m_ago(모멘텀 계산용 스냅샷)

In [3]:
prices.groupBy("snapshot_type", "bas_dt").count().orderBy("snapshot_type", "bas_dt").toPandas()

,snapshot_type,bas_dt,count
0,12m_ago,20201230,2081
1,12m_ago,20211230,2170
2,12m_ago,20221229,2245
3,12m_ago,20250724,2489
4,12m_ago,20250729,2492
...,...,...,...
812,current,20260722,2554
813,current,20260723,2554
814,current,20260724,2555
815,current,20260727,2555


In [4]:
# 특정 종목 시세 흐름 확인 (종목코드 바꿔가며 조회)
prices.filter(F.col("stock_code") == "005930").orderBy("snapshot_type", "bas_dt").toPandas()

,bas_dt,close_price,fluctuation_rate,listed_share_count,market_cap,open_price,snapshot_type,stock_code,year
0,20201230,81000,3.45,5969782550,483552386550000,77400,12m_ago,005930,2020
1,20211230,78300,-0.63,5969782550,467433973665000,78900,12m_ago,005930,2021
2,20221229,55300,-2.30,5969782550,330128975015000,56000,12m_ago,005930,2022
3,20250724,66000,-0.60,5919637922,390696102852000,66500,12m_ago,005930,2025
4,20250729,70600,0.28,5919637922,417926437293200,70800,12m_ago,005930,2025
...,...,...,...,...,...,...,...,...,...
812,20260722,260500,0.58,5846278608,1522955577384000,276000,current,005930,2026
813,20260723,270000,3.65,5846278608,1578495224160000,269000,current,005930,2026
814,20260724,249500,-7.59,5846278608,1458646512696000,266000,current,005930,2026
815,20260727,254000,1.80,5846278608,1484954766432000,257000,current,005930,2026


## 3. financials (재무제표, --dart-limit 적용 종목만 존재)

In [5]:
print(f"재무제표 보유 종목 수: {financials.select('stock_code').distinct().count()}개")
print(f"전체 상장 종목 수: {companies.select('stock_code').distinct().count()}개")

financials.groupBy("fs_div", "currency").count().orderBy("fs_div", "currency").toPandas()

재무제표 보유 종목 수: 2368개
전체 상장 종목 수: 2555개


,fs_div,currency,count
0,CFS,CNY,5604
1,CFS,HKD,264
2,CFS,JPY,737
3,CFS,KRW,1304173
4,CFS,USD,4263
5,OFS,KRW,223878
6,OFS,USD,940


In [6]:
# 특정 종목 재무제표 항목 확인 (종목코드 바꿔가며 조회)
financials.filter(F.col("stock_code") == "000020") \
    .select("account_id", "account_nm", "sj_nm", "thstrm_nm", "thstrm_amount", "frmtrm_nm", "frmtrm_amount") \
    .toPandas()

,account_id,account_nm,sj_nm,thstrm_nm,thstrm_amount,frmtrm_nm,frmtrm_amount
0,ifrs-full_Assets,자산총계,재무상태표,제 95 기,564972408621,제 94 기,461136459052
1,ifrs-full_CurrentAssets,유동자산,재무상태표,제 95 기,237700038220,제 94 기,227527470560
2,ifrs-full_CashAndCashEquivalents,현금및현금성자산,재무상태표,제 95 기,72317114790,제 94 기,34347103567
3,ifrs-full_CurrentTaxAssets,당기법인세자산,재무상태표,제 95 기,532183,제 94 기,3018897413
4,ifrs-full_Inventories,재고자산,재무상태표,제 95 기,70697183860,제 94 기,46783642034
...,...,...,...,...,...,...,...
986,ifrs-full_Equity,기말자본,자본변동표,제 92 기,342682240439,제 91 기,299708392624
987,ifrs-full_Equity,기말자본,자본변동표,제 92 기,11426725883,제 91 기,0
988,ifrs-full_Equity,기말자본,자본변동표,제 92 기,331255514556,제 91 기,299708392624
989,ifrs-full_Equity,기말자본,자본변동표,제 92 기,-2521453524,제 91 기,-2521453524


## 4. dividends (배당, --dart-limit 적용 종목만 존재)

In [7]:
dividends.groupBy("se", "stock_knd").count().orderBy("se", "stock_knd").toPandas()

,se,stock_knd,count
0,(별도)당기순이익(백만원),-,9009
1,(연결)당기순이익(백만원),-,9009
2,(연결)주당순이익(원),-,9009
3,(연결)현금배당성향(%),-,9009
4,주당 주식배당(주),보통주,1
...,...,...,...
242,현금배당수익률(%),제3우선주,4
243,현금배당수익률(%),제4우선주식(*),1
244,현금배당수익률(%),종류주,108
245,현금배당수익률(%),종류주식,178


In [8]:
spark.stop()